In [13]:
!pip install backtesting --quiet
!pip install yfinance --quiet
!pip install pandas_ta --quiet

In [17]:
try:
  import pandas as pd
  import numpy as np
  import yfinance as yf
  import pandas_ta as ta
  from backtesting import Backtest, Strategy
  from backtesting.lib import crossover
  from backtesting.test import SMA
  print("Libraries successfully imported!")
except:
  print("Error importing libraries")

Libraries successfully imported!


In [18]:
df = yf.Ticker('NVDA').history(period='max')

In [19]:
def RSI(close, length):
    return ta.rsi(pd.Series(close), length=length)

def MACD_line(close, fast, slow, signal):
    m = ta.macd(pd.Series(close), fast=fast, slow=slow, signal=signal)
    return m.iloc[:, 0]

def MACD_signal(close, fast, slow, signal):
    m = ta.macd(pd.Series(close), fast=fast, slow=slow, signal=signal)
    return m.iloc[:, 2]

def BB_lower(close, length, std):
    return ta.bbands(pd.Series(close), length=length, std=std).iloc[:, 0]

def BB_upper(close, length, std):
    return ta.bbands(pd.Series(close), length=length, std=std).iloc[:, 2]


class CustomStrategy(Strategy):
    sma_fast    = 10
    sma_slow    = 50
    rsi_len     = 14
    rsi_max_buy = 70     # not buying if overbought
    rsi_exit    = 75     # exit if RSI is high
    bb_len      = 20
    bb_std      = 2
    macd_fast   = 12
    macd_slow   = 26
    macd_sig    = 9

    def init(self):
        c = self.data.Close

        self.sma_f = self.I(SMA, c, self.sma_fast)
        self.sma_s = self.I(SMA, c, self.sma_slow)

        self.rsi    = self.I(RSI,         c, self.rsi_len)
        self.macd   = self.I(MACD_line,   c, self.macd_fast, self.macd_slow, self.macd_sig)
        self.signal = self.I(MACD_signal, c, self.macd_fast, self.macd_slow, self.macd_sig)

        self.bb_lo = self.I(BB_lower, c, self.bb_len, self.bb_std)
        self.bb_hi = self.I(BB_upper, c, self.bb_len, self.bb_std)

    def next(self):
        price = self.data.Close[-1]

        # Buy conditions
        trend_up = self.sma_f[-1] > self.sma_s[-1]
        macd_up  = crossover(self.macd, self.signal)
        rsi_ok   = self.rsi[-1] < self.rsi_max_buy

        # Exit conditions
        macd_down = crossover(self.signal, self.macd)
        rsi_hot   = self.rsi[-1] > self.rsi_exit
        hit_upper = price >= self.bb_hi[-1]

        if not self.position:
            if trend_up and macd_up and rsi_ok:
                self.buy()
        else:
            if macd_down or rsi_hot or hit_upper:
                self.position.close()

In [20]:
bt = Backtest(df, CustomStrategy, cash=10_000, commission=0.002)
stats = bt.run()
print(stats)
bt.plot()

Backtest.run:   0%|          | 0/6842 [00:00<?, ?bar/s]

Start                     1999-01-22 00:00...
End                       2026-06-16 00:00...
Duration                  10006 days 23:00:00
Exposure Time [%]                     7.39988
Equity Final [$]                  11728.09192
Equity Peak [$]                   18100.91891
Commissions [$]                    6514.20111
Return [%]                           17.28092
Buy & Hold Return [%]            546922.04284
Return (Ann.) [%]                     0.58454
Volatility (Ann.) [%]                11.28321
CAGR [%]                              0.40222
Sharpe Ratio                          0.05181
Sortino Ratio                         0.08041
Calmar Ratio                          0.01611
Alpha [%]                        -18423.91641
Beta                                  0.03372
Max. Drawdown [%]                   -36.27318
Avg. Drawdown [%]                    -8.94068
Max. Drawdown Duration     4688 days 00:00:00
Avg. Drawdown Duration      583 days 00:00:00
# Trades                          

/usr/local/lib/python3.12/dist-packages/bokeh/util/serialization.py:242: UserWarning: no explicit representation of timezones available for np.datetime64
  return convert(array.astype("datetime64[us]"))


GridPlot(id='p1524', ...)